# Запуск контейнера Postgres
С проброшенными портами для локального доступа и заданным логином и паролем суперюзера.

```bash
docker run -e POSTGRES_PASSWORD=postgres -e POSTGRES_USER=postgres -p 5432:5432 -d postgres
```

## Создать таблицу с `id`
```sql
CREATE TABLE accounts(
id SERIAL PRIMARY KEY);
```

## Добавить столбцы у таблицы
```sql
ALTER TABLE accounts
ADD COLUMN user_id VARCHAR NOT NULL UNIQUE,
ADD COLUMN balance DECIMAL(10, 2) NOT NULL,
ADD COLUMN updated_time TIMESTAMP DEFAULT NOW(),
ADD COLUMN extra VARCHAR;
```

## Удалить столбцы у таблицы
```sql
ALTER TABLE accounts DROP COLUMN extra;
```

## Обновить название столбца
```sql
ALTER TABLE accounts RENAME COLUMN updated_time TO updated;
```

## CRUD
### Insert
```sql
INSERT INTO accounts (user_id, balance)
VALUES
(1, 100),
(2, 0);
```

### Update
```sql
UPDATE accounts SET balance = 0 WHERE user_id = 1
```

### Delete
```sql
DELETE FROM accounts where balance = 0;
```

## Генерация тестовых данных
```sql
INSERT INTO accounts (user_id, amount)
SELECT
    substr(md5(random()::text)),  -- Генерируем случайную строку длиной в 32 символа
    floor(random() * 1000000)  -- Генерируем случайное число 0 - 1000000
FROM
    generate_series(1, 1000); -- Функция для вычислений в циклах 1000 раз
```

## Дедлок
Эмулируем. Открываем два окна Бобра. Делаем команды:
```sql
-- Window 1
BEGIN;

SELECT user_id FROM accounts WHERE user_id = 'a' FOR UPDATE;
SELECT user_id FROM accounts WHERE user_id = 'b' FOR UPDATE;
```

```sql
--- Window 2, меняем местами запрашиваемые строки
BEGIN;

SELECT user_id FROM accounts WHERE user_id = 'b' FOR UPDATE;
SELECT user_id FROM accounts WHERE user_id = 'a' FOR UPDATE;
```

Получим вывод
```sql
SQL Error [40P01]: ERROR: deadlock detected
  Detail: Process 415 waits for ShareLock on transaction 794; blocked by process 416.
Process 416 waits for ShareLock on transaction 793; blocked by process 415.
  Hint: See server log for query details.
  Where: while locking tuple (20,63) in relation "accounts"

Error position:
```

### Исследование дедлока
В отдельной сессии смотрим зависшие транзакции.
```sql
SELECT
    pid,
    now() - pg_stat_activity.query_start AS duration,
    query,
    state,
    wait_event_type,
    wait_event,
    usename,
    application_name
FROM pg_stat_activity
WHERE (now() - pg_stat_activity.query_start) > interval '30 seconds'
  AND state != 'idle'
ORDER BY duration DESC;
```
Увидим одну из транзакций с `wait_event_type` == `Lock`, либо, если из Бобра, `Client`.

### Кто кого заблокировал
```sql
SELECT
    blocked_locks.pid AS blocked_pid,
    blocked_activity.usename AS blocked_user,
    blocking_locks.pid AS blocking_pid,
    blocking_activity.usename AS blocking_user,
    blocked_activity.query AS blocked_query,
    blocking_activity.query AS blocking_query,
    blocked_locks.locktype,
    now() - blocked_activity.query_start AS blocked_duration
FROM pg_catalog.pg_locks blocked_locks
JOIN pg_catalog.pg_stat_activity blocked_activity
    ON blocked_activity.pid = blocked_locks.pid
JOIN pg_catalog.pg_locks blocking_locks
    ON blocking_locks.locktype = blocked_locks.locktype
    AND blocking_locks.database IS NOT DISTINCT FROM blocked_locks.database
    AND blocking_locks.relation IS NOT DISTINCT FROM blocked_locks.relation
    AND blocking_locks.page IS NOT DISTINCT FROM blocked_locks.page
    AND blocking_locks.tuple IS NOT DISTINCT FROM blocked_locks.tuple
    AND blocking_locks.virtualxid IS NOT DISTINCT FROM blocked_locks.virtualxid
    AND blocking_locks.transactionid IS NOT DISTINCT FROM blocked_locks.transactionid
    AND blocking_locks.classid IS NOT DISTINCT FROM blocked_locks.classid
    AND blocking_locks.objid IS NOT DISTINCT FROM blocked_locks.objid
    AND blocking_locks.objsubid IS NOT DISTINCT FROM blocked_locks.objsubid
    AND blocking_locks.pid != blocked_locks.pid
JOIN pg_catalog.pg_stat_activity blocking_activity
    ON blocking_activity.pid = blocking_locks.pid
WHERE NOT blocked_locks.granted
ORDER BY blocked_duration DESC;
```